# Solutions — Selecting, Filtering, Sorting, Top-N, Latest Versions

Exercise notebook: `09_pandas/02_filtering_selecting_sorting.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

### Setup (same as the exercise notebook)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
pd.set_option("display.width", 140)
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
orders_raw = pd.read_csv(DATA / "csv" / "orders.csv", dtype=str, keep_default_na=False)
print(orders_raw.shape); orders_raw.head()

## Exercise 1 — 🟢 Easy · Boolean masks and column selection

**Working solution**

In [ ]:
def big_web_orders(df, min_amount):
    mask = (df["channel"] == "web") & (df["amount"] >= min_amount) & (df["status"] != "CANCELLED")
    return (df.loc[mask, ["order_id", "customer_id", "amount"]]
              .sort_values(["amount", "order_id"], ascending=[False, True])
              .reset_index(drop=True))

**Explanation**

Combine masks with `&`, `|`, `~` and **parentheses** (Python's `and`/`or` don't work element-wise). Method chaining in parentheses reads like a SQL query.

**Complexity:** O(n log n).

**Common mistakes**

- `df[df.channel == 'web' and ...]` → `ValueError: truth value of a Series is ambiguous`.
- Forgetting `reset_index(drop=True)`.

## Exercise 2 — 🟡 Medium · isin, between and date ranges

**Working solution**

In [ ]:
import pandas as pd

def query_orders(df, statuses, countries, start, end):
    mask = df["status"].isin(statuses) & df["order_date"].between(pd.Timestamp(start), pd.Timestamp(end))
    if countries is not None:
        mask &= df["country"].isin(countries)
    return df.loc[mask]

**Explanation**

Build the mask incrementally with `&=` for optional filters — no `if/else` explosion of query variants.

**Complexity:** O(n).

**Common mistakes**

- Comparing a datetime column with a string inside `between` in old pandas versions — be explicit with `pd.Timestamp`.

## Exercise 3 — 🟡 Medium · Top-N per group

**Working solution**

In [ ]:
def top_n_per_group(df, group, value, n=2):
    return (df.sort_values([group, value, "order_id"], ascending=[True, False, True])
              .groupby(group, sort=False)
              .head(n)
              .reset_index(drop=True))

**Explanation**

Sort once, then `groupby().head(n)` keeps the first n rows per group in the existing order — the pandas form of `ROW_NUMBER() <= n`.

**Complexity:** O(n log n).

**Common mistakes**

- `groupby().apply(lambda g: g.nlargest(n))` — slower and adds a confusing extra index level.

## Exercise 4 — 🟠 Hard · Latest version per key

**Working solution**

In [ ]:
def latest_versions(df, key="order_id", version="updated_at"):
    ordered = df.assign(_pos=range(len(df))).sort_values([version, "_pos"], na_position="first", kind="stable")
    latest = ordered.drop_duplicates(subset=key, keep="last").sort_values(key).drop(columns="_pos").reset_index(drop=True)
    latest.attrs["dropped"] = len(df) - len(latest)
    return latest

**Explanation**

`sort_values` + `drop_duplicates(keep="last")` is the vectorized 'latest wins'. A positional helper column makes the tie-break explicit and stable.
`df.attrs` is a dict for metadata that travels with the frame (experimental, but handy).

**Complexity:** O(n log n).

**Common mistakes**

- Sorting by version only and getting arbitrary tie-breaks.
- NaT sorting last by default → a row with a missing timestamp wins.

## Challenge — 🔴 Very Hard · Declarative rule filtering

**Working solution**

In [ ]:
import pandas as pd

def apply_rules(df, rules):
    results = pd.DataFrame({name: df.eval(expr).fillna(False).astype(bool) for name, expr in rules.items()}, index=df.index)
    passed = results.all(axis=1) if rules else pd.Series(True, index=df.index)
    failed_names = (~results).apply(lambda row: ",".join(name for name, failed in row.items() if failed), axis=1) if rules else pd.Series("", index=df.index)
    return df.loc[passed], df.loc[~passed].assign(failed_rules=failed_names[~passed])

**Explanation**

Evaluating each rule into a boolean column gives a rule × row matrix: `all(axis=1)` = passes everything, and the failed names are a row-wise reduction. `apply(axis=1)` is OK here
because it runs only on the (usually small) set of rule columns; never use it for arithmetic.

**Complexity:** O(n × rules).

**Common mistakes**

- `df.query()` per rule and set-subtracting indices — loses which rule failed.
- Comparisons against NaN silently evaluate False — be explicit about it.